# Analysis of 1000 Genomes Genomic Copy Number Variations (CNVs)

This notebook demonstrates how to access, query, and analyze genomic CNV datasets managed by LaminDB. It covers:
1. Querying validated CNV artifacts via schemas.
2. Accessing datasets lazily as PyArrow datasets.
3. Computing sample-level summary statistics (deletions, duplications, zygosity).
4. Visualizing CNV distribution across samples.
5. Identifying recurrent CNVs across samples.

In [ ]:
import lamindb as ln
import pyarrow as pa
import pyarrow.compute as pc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Track the notebook run using LaminDB
ln.track(project="1000 Genomes")

## 1. Access and Inspect CNV Datasets

Query the metadata database to locate all validated CNV datasets for the "1000 Genomes" project.

In [ ]:
# Retrieve the schema used for CNV validation
schema = ln.Schema.get(name="1000 Genomes CNV VCF")
projects = ln.Project.lookup()

# Filter and display artifacts matching the schema and project
artifacts = ln.Artifact.filter(
    schema=schema, 
    projects=projects.ln_1000_genomes
).order_by("created_at")

artifacts.to_dataframe()

### Describe Selected Artifact and Schema

Let's inspect the metadata and the validating schema structure of the first artifact.

In [ ]:
artifact = artifacts[0]

print("--- Artifact Metadata ---")
artifact.describe()

print("\n--- Validating Schema ---")
artifact.schema.describe()

## 2. Load Data as PyArrow Dataset

Open the artifacts lazily as a unified PyArrow dataset to perform efficient streaming and filtering operations without downloading all files into memory.

In [ ]:
dataset = artifacts.open()
dataset

## 3. Compute Sample-Level CNV Statistics

Calculate sample-specific statistics, including deletion/duplication counts, size statistics, and zygosity (homozygous vs. heterozygous) counts.

In [ ]:
# Retrieve unique samples
sample_names = dataset.to_table(columns=["SAMPLE_NAME"]).to_pandas()["SAMPLE_NAME"].unique()
print(f"Analyzing {len(sample_names)} samples...")

sample_stats = []

# Process each sample
for sample_name in sample_names:
    # Filter dataset efficiently using PyArrow
    sample_filter = pc.equal(pc.field("SAMPLE_NAME"), pa.scalar(sample_name))
    sample_ds = dataset.filter(sample_filter)
    
    total_cnvs = sample_ds.count_rows()
    sample_df = sample_ds.to_table().to_pandas()
    
    # Deletions (negative lengths) vs Duplications
    deletions = sample_df[sample_df["INFO_SVLEN"] < 0]
    deletion_count = len(deletions)
    duplication_count = total_cnvs - deletion_count
    
    # Size stats for deletions
    min_del_size = abs(deletions["INFO_SVLEN"].min()) if not deletions.empty else 0
    max_del_size = abs(deletions["INFO_SVLEN"].max()) if not deletions.empty else 0
    median_del_size = abs(deletions["INFO_SVLEN"].median()) if not deletions.empty else 0
    
    # Genotype (homozygous "1/1" vs heterozygous "0/1")
    homo_count = sum(sample_df["SAMPLE_GT"] == "1/1")
    hetero_count = sum(sample_df["SAMPLE_GT"] == "0/1")
    
    # High confidence segment mean values
    high_conf_cnvs = None
    if "SAMPLE_SM" in sample_df.columns:
        high_conf_cnvs = sum(abs(sample_df["SAMPLE_SM"] - 1.0) > 0.5)
        
    sample_stats.append({
        "Sample": sample_name,
        "Total_CNVs": total_cnvs,
        "Deletions": deletion_count,
        "Duplications": duplication_count,
        "Min_Deletion_Size": min_del_size,
        "Max_Deletion_Size": max_del_size,
        "Median_Deletion_Size": median_del_size,
        "Homozygous_CNVs": homo_count,
        "Heterozygous_CNVs": hetero_count,
        "High_Confidence_CNVs": high_conf_cnvs
    })

pd.DataFrame(sample_stats)

### Visualize CNV Distribution Across Samples

Generate a bar plot comparing the number of deletions vs. duplications for each sample.

In [ ]:
# Set styling
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))

# Reshape the data for plotting
stats_melted = stats_df.melt(
    id_vars="Sample", 
    value_vars=["Deletions", "Duplications"], 
    var_name="CNV_Type", 
    value_name="Count"
)

sns.barplot(data=stats_melted, x="Sample", y="Count", hue="CNV_Type", palette="muted", ax=ax)
ax.set_title("CNV Deletions vs. Duplications per Sample", fontsize=14, fontweight="bold")
ax.set_xlabel("Sample Name", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Identify Recurrent CNVs Across Samples

Locate recurrent genomic regions harboring CNVs across multiple individuals. We group CNVs into genomic bins of a specified size (e.g., 1000 bp) and identify bins with variations present in multiple samples.

In [ ]:
def find_recurrent_cnvs(dataset, proximity_threshold=1000, min_samples=2):
    """Identify genomic bins containing CNVs in multiple samples."""
    cnvs_table = dataset.to_table(columns=["CHROM", "POS", "INFO_SVLEN", "SAMPLE_NAME"])
    cnvs_df = cnvs_table.to_pandas()
    
    # Calculate End Position
    cnvs_df["INFO_END"] = cnvs_df.apply(
        lambda r: r["POS"] + abs(r["INFO_SVLEN"]) if pd.notna(r["INFO_SVLEN"]) else r["POS"],
        axis=1
    )
    
    # Define binning
    bin_size = proximity_threshold
    cnvs_df["bin_start"] = (cnvs_df["POS"] // bin_size) * bin_size
    cnvs_df["bin_end"] = (cnvs_df["INFO_END"].astype(float) // bin_size) * bin_size
    cnvs_df["region_key"] = cnvs_df["CHROM"] + ":" + cnvs_df["bin_start"].astype(str)
    
    # Group to find recurrent regions
    region_counts = cnvs_df.groupby("region_key")["SAMPLE_NAME"].nunique().reset_index()
    region_counts.columns = ["region_key", "sample_count"]
    
    recurrent_regions = region_counts[region_counts["sample_count"] >= min_samples]
    print(f"Found {len(recurrent_regions)} recurrent CNV regions in >= {min_samples} samples.")
    
    if recurrent_regions.empty:
        return pd.DataFrame()
        
    recurrent_cnvs = pd.merge(
        cnvs_df, 
        recurrent_regions[["region_key", "sample_count"]], 
        on="region_key"
    )
    
    recurrent_summary = []
    for region, group in recurrent_cnvs.groupby("region_key"):
        chrom = group["CHROM"].iloc[0]
        start = group["POS"].min()
        end = group["INFO_END"].max()
        samples = group["SAMPLE_NAME"].unique()
        
        sizes = group["INFO_SVLEN"].abs().dropna()
        min_size = sizes.min() if not sizes.empty else None
        max_size = sizes.max() if not sizes.empty else None
        median_size = sizes.median() if not sizes.empty else None
        
        recurrent_summary.append({
            "CHROM": chrom,
            "START": start,
            "END": end,
            "LENGTH": end - start,
            "NUM_SAMPLES": len(samples),
            "SAMPLES": ", ".join(samples),
            "NUM_CNVS": len(group),
            "MIN_SIZE": min_size,
            "MAX_SIZE": max_size,
            "MEDIAN_SIZE": median_size
        })
        
    recurrent_df = pd.DataFrame(recurrent_summary)
    recurrent_df = recurrent_df.sort_values(["NUM_SAMPLES", "NUM_CNVS"], ascending=False)
    return recurrent_df

recurrent_df = find_recurrent_cnvs(dataset)
recurrent_df.head(20)

## 5. Finish Tracking

Mark the pipeline tracking run as completed in LaminDB.

In [ ]:
ln.finish()